In [ ]:
# ============================================================
#  11 Composite Analysis
#  汇聚所有上游数据 → demand × friction overlap → 选址 → 策略
#  输入: 08 POI + 09 Population + 10 OD/Friction + 07 Compounds
#  输出: gap index / candidate sites / strategy maps
# ============================================================

from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Point, box
from shapely.ops import unary_union
from shapely.prepared import prep as shapely_prep
from scipy.spatial import cKDTree
import warnings
warnings.filterwarnings("ignore")

OUT = Path("data_out")
OUT.mkdir(exist_ok=True)

# ── 上游数据路径 ──
BOUNDARY     = Path("../04 Transport/data_raw/shenzhen_boundary.geojson")
OD_ANALYSIS  = Path("../10 OD & Ground Friction/data_out/sz_od_analysis.gpkg")
ROUTES       = Path("../10 OD & Ground Friction/data_out/sz_routes.gpkg")
DEMAND_GRID  = Path("../08 POI Demand/data_out/sz_demand_grid.gpkg")
POP_GRID     = Path("../09 Population/data_out/sz_population_grid.gpkg")
COMPOUNDS    = Path("../07 Parks & Compounds/data_out/sz_compounds.gpkg")
BLDG_GRID    = Path("../06 Buildings/data_out/sz_building_grid.gpkg")
BARRIERS     = Path("../05 Barrier Layers/data_out/sz_barrier_unions.gpkg")

# ── 加载 ──
shenzhen = gpd.read_file(BOUNDARY).to_crs(4326)
shenzhen_geom = shenzhen.union_all() if hasattr(shenzhen, "union_all") else shenzhen.unary_union

print("Loading upstream data ...")
od = gpd.read_file(OD_ANALYSIS) if OD_ANALYSIS.exists() else None
demand = gpd.read_file(DEMAND_GRID) if DEMAND_GRID.exists() else None
pop = gpd.read_file(POP_GRID) if POP_GRID.exists() else None
compounds = gpd.read_file(COMPOUNDS) if COMPOUNDS.exists() else None
bldg_grid = gpd.read_file(BLDG_GRID) if BLDG_GRID.exists() else None

for name, df in [("OD", od), ("Demand", demand), ("Pop", pop),
                  ("Compounds", compounds), ("BldgGrid", bldg_grid)]:
    if df is not None:
        print(f"  {name:12s} {len(df):>8,} rows")
    else:
        print(f"  {name:12s} NOT FOUND")

In [ ]:
# ============================================================
#  1. Demand × Friction Overlap → Gap Index
#  在统一网格上合并 demand_pressure 和 ground_friction
#  gap_index = demand × friction → 高需求 + 高摩擦 = 无人机黄金区域
# ============================================================

GRID_SIZE = 0.005  # 与 06/08/09 一致

# ── 构建统一网格 ──
minx, miny, maxx, maxy = shenzhen_geom.bounds
sz_prep = shapely_prep(shenzhen_geom)

grid_cells = []
gid = 0
x = minx
while x < maxx:
    y = miny
    while y < maxy:
        cell = box(x, y, x + GRID_SIZE, y + GRID_SIZE)
        if sz_prep.intersects(cell):
            grid_cells.append({"grid_id": gid, "cx": x + GRID_SIZE/2, "cy": y + GRID_SIZE/2, "geometry": cell})
            gid += 1
        y += GRID_SIZE
    x += GRID_SIZE

grid = gpd.GeoDataFrame(grid_cells, crs=4326)
print(f"Unified grid: {len(grid):,} cells")

# ── 合并 demand_pressure (from 08) ──
if demand is not None and "demand_pressure" in demand.columns:
    demand_pts = demand.copy()
    demand_pts["geometry"] = demand_pts.geometry.centroid
    dj = gpd.sjoin_nearest(
        grid[["grid_id", "geometry"]].copy().set_geometry(grid.geometry.centroid),
        demand_pts[["geometry", "demand_pressure"]],
        how="left", max_distance=0.01,
    )
    grid["demand_pressure"] = dj["demand_pressure"].fillna(0).values
    # 也取各类 count
    for col in [c for c in demand.columns if c.endswith("_count")]:
        dj2 = gpd.sjoin_nearest(
            grid[["grid_id", "geometry"]].copy().set_geometry(grid.geometry.centroid),
            demand_pts[["geometry", col]],
            how="left", max_distance=0.01,
        )
        grid[col] = dj2[col].fillna(0).values
    print(f"  demand_pressure merged")
else:
    grid["demand_pressure"] = 0
    print("  demand_pressure: NOT AVAILABLE")

# ── 合并 intensity_index (from 09) ──
if pop is not None and "intensity_index" in pop.columns:
    pop_pts = pop.copy()
    pop_pts["geometry"] = pop_pts.geometry.centroid
    pj = gpd.sjoin_nearest(
        grid[["grid_id", "geometry"]].copy().set_geometry(grid.geometry.centroid),
        pop_pts[["geometry", "intensity_index", "pop_count"]],
        how="left", max_distance=0.01,
    )
    grid["intensity_index"] = pj["intensity_index"].fillna(0).values
    grid["pop_count"] = pj["pop_count"].fillna(0).values
    print(f"  intensity_index merged")
else:
    grid["intensity_index"] = 0
    grid["pop_count"] = 0
    print("  intensity_index: NOT AVAILABLE")

# ── 合并 ground_friction (from 10, 聚合到网格) ──
if od is not None and "ground_friction" in od.columns:
    od_pts_o = gpd.GeoDataFrame(
        od[["ground_friction", "detour_ratio", "congestion_amplifier"]],
        geometry=gpd.points_from_xy(od["o_lon"], od["o_lat"]), crs=4326,
    )
    oj = gpd.sjoin(od_pts_o, grid[["grid_id", "geometry"]], how="inner", predicate="within")
    friction_by_grid = oj.groupby("grid_id").agg(
        avg_friction=("ground_friction", "mean"),
        avg_detour=("detour_ratio", "mean"),
        avg_congestion=("congestion_amplifier", "mean"),
    ).reset_index()
    grid = grid.merge(friction_by_grid, on="grid_id", how="left")
    grid[["avg_friction", "avg_detour", "avg_congestion"]] = \
        grid[["avg_friction", "avg_detour", "avg_congestion"]].fillna(0)
    print(f"  ground_friction merged (from {len(friction_by_grid)} grids)")
else:
    grid["avg_friction"] = 0
    grid["avg_detour"] = 0
    grid["avg_congestion"] = 0
    print("  ground_friction: NOT AVAILABLE")

# ── 计算 Gap Index ──
def norm(s):
    mn, mx = s.min(), s.max()
    return (s - mn) / (mx - mn) if mx > mn else s * 0

grid["demand_norm"] = norm(grid["demand_pressure"])
grid["friction_norm"] = norm(grid["avg_friction"])
grid["intensity_norm"] = norm(grid["intensity_index"])

# gap = demand × friction (交叉乘积, 两者都高才高)
grid["gap_index"] = (
    0.4 * grid["demand_norm"] * grid["friction_norm"]
  + 0.3 * grid["intensity_norm"] * grid["friction_norm"]
  + 0.3 * grid["demand_norm"] * grid["intensity_norm"]
)

non_zero = grid["gap_index"] > 0
print(f"\n=== Gap Index ===")
print(f"Grids with gap > 0: {non_zero.sum():,} / {len(grid):,}")
print(f"Gap: mean={grid.loc[non_zero, 'gap_index'].mean():.4f}  max={grid['gap_index'].max():.4f}")
print(f"\nTop 10 gap grids:")
print(grid.nlargest(10, "gap_index")[["grid_id", "demand_pressure", "avg_friction", "intensity_index", "gap_index"]].to_string(index=False))

In [ ]:
# ============================================================
#  2. Candidate Site Pool (候选起降点)
#  筛选逻辑:
#    - gap_index 高的网格中心 (需求大 + 地面难)
#    - 不在水体/铁路 barrier 内
#    - 靠近 compound (有屋顶/空地可用)
#    - 按 gap_index 排名, 保留 top N
# ============================================================

# 排除 barrier 区域
barrier_unions = gpd.read_file(BARRIERS) if BARRIERS.exists() else None
exclude_geom = None
if barrier_unions is not None:
    water_geom = barrier_unions[barrier_unions["barrier_type"] == "water"].geometry
    if len(water_geom) > 0:
        exclude_geom = unary_union(water_geom)
        exclude_prep = shapely_prep(exclude_geom)

# 从高 gap 网格中选候选点
TOP_N = 200
candidates = grid[grid["gap_index"] > 0].nlargest(TOP_N * 3, "gap_index").copy()
candidates["candidate_point"] = candidates.apply(
    lambda r: Point(r["cx"], r["cy"]), axis=1
)

# 过滤: 排除落在水体内的
if exclude_geom is not None:
    candidates["in_water"] = candidates["candidate_point"].apply(lambda pt: exclude_prep.contains(pt))
    candidates = candidates[~candidates["in_water"]].drop(columns=["in_water"])

# 过滤: 与已选候选点保持最小间距 (避免扎堆)
MIN_SPACING_DEG = 0.008  # ~900m
selected = []
selected_coords = []
for _, row in candidates.iterrows():
    pt = (row["cx"], row["cy"])
    too_close = False
    for sc in selected_coords:
        if abs(pt[0] - sc[0]) < MIN_SPACING_DEG and abs(pt[1] - sc[1]) < MIN_SPACING_DEG:
            too_close = True
            break
    if not too_close:
        selected.append(row)
        selected_coords.append(pt)
    if len(selected) >= TOP_N:
        break

site_pool = gpd.GeoDataFrame(selected, crs=4326)
site_pool["geometry"] = site_pool["candidate_point"]
site_pool = site_pool.drop(columns=["candidate_point"]).reset_index(drop=True)
site_pool["site_id"] = range(len(site_pool))

# 为每个候选点标注最近 compound 类型
if compounds is not None:
    comp_centroids = np.array(list(zip(
        compounds.geometry.centroid.x, compounds.geometry.centroid.y
    )))
    comp_tree = cKDTree(comp_centroids)
    dists, idxs = comp_tree.query(np.array(list(zip(site_pool["cx"], site_pool["cy"]))))
    site_pool["nearest_compound_type"] = compounds.iloc[idxs]["compound_type"].values
    site_pool["nearest_compound_dist_m"] = dists * 111320

# 分类: hub / station / endpoint
site_pool["site_class"] = pd.cut(
    site_pool["gap_index"],
    bins=[0, site_pool["gap_index"].quantile(0.33),
          site_pool["gap_index"].quantile(0.66), 1],
    labels=["endpoint", "station", "hub"],
)

print(f"Candidate sites: {len(site_pool)}")
print(f"\nBy class:")
print(site_pool["site_class"].value_counts().to_string())
print(f"\nBy nearest compound:")
print(site_pool["nearest_compound_type"].value_counts().to_string())

In [ ]:
# ============================================================
#  3. +3 / +5 / +10 Strategy Simulation
#  从候选池中选 top-K 个站点, 计算覆盖范围和收益
#  覆盖 = 3km 缓冲圆内的 demand_pressure 总量
# ============================================================

COVERAGE_RADIUS_DEG = 3000 / 111320  # 3km
STRATEGIES = [3, 5, 10]

# 预计算每个网格中心坐标
grid_coords = np.array(list(zip(grid["cx"], grid["cy"])))
grid_tree = cKDTree(grid_coords)

def evaluate_strategy(sites_gdf, n_sites, grid_df, grid_tree, radius):
    """选 top-n 站点, 计算覆盖率和总需求"""
    top = sites_gdf.head(n_sites)

    covered_grids = set()
    for _, site in top.iterrows():
        idxs = grid_tree.query_ball_point([site["cx"], site["cy"]], radius)
        covered_grids.update(idxs)

    total_demand = grid_df["demand_pressure"].sum()
    covered_demand = grid_df.iloc[list(covered_grids)]["demand_pressure"].sum()
    total_pop = grid_df["pop_count"].sum()
    covered_pop = grid_df.iloc[list(covered_grids)]["pop_count"].sum()

    coverage_area_pct = len(covered_grids) / len(grid_df) * 100
    demand_coverage_pct = covered_demand / total_demand * 100 if total_demand > 0 else 0
    pop_coverage_pct = covered_pop / total_pop * 100 if total_pop > 0 else 0

    # 生成覆盖圈 polygon
    buffers = top.geometry.buffer(radius)
    coverage_union = unary_union(buffers)

    return {
        "n_sites": n_sites,
        "covered_grids": len(covered_grids),
        "coverage_area_pct": coverage_area_pct,
        "demand_coverage_pct": demand_coverage_pct,
        "pop_coverage_pct": pop_coverage_pct,
        "covered_demand": covered_demand,
        "covered_pop": covered_pop,
        "site_ids": list(top["site_id"]),
        "coverage_geom": coverage_union,
    }

# 按 gap_index 排序 (hub 优先)
site_pool_sorted = site_pool.sort_values("gap_index", ascending=False).reset_index(drop=True)

strategy_results = []
print("=== Strategy Simulation (3km coverage) ===\n")
print(f"{'Strategy':>10s} {'Sites':>6s} {'Area%':>7s} {'Demand%':>8s} {'Pop%':>7s}")
print("-" * 42)

for n in STRATEGIES:
    r = evaluate_strategy(site_pool_sorted, n, grid, grid_tree, COVERAGE_RADIUS_DEG)
    strategy_results.append(r)
    print(f"  +{n:<8d} {r['n_sites']:>6d} {r['coverage_area_pct']:>6.1f}% {r['demand_coverage_pct']:>7.1f}% {r['pop_coverage_pct']:>6.1f}%")

# 也算 +20, +50 看边际收益
for n in [20, 50]:
    if n <= len(site_pool_sorted):
        r = evaluate_strategy(site_pool_sorted, n, grid, grid_tree, COVERAGE_RADIUS_DEG)
        strategy_results.append(r)
        print(f"  +{n:<8d} {r['n_sites']:>6d} {r['coverage_area_pct']:>6.1f}% {r['demand_coverage_pct']:>7.1f}% {r['pop_coverage_pct']:>6.1f}%")

In [ ]:
# ============================================================
#  4. Relief Vulnerability Map + 保存所有输出
#  relief_vulnerability = gap_index × (1 - coverage)
#  未被任何策略覆盖 + gap 高 → 最需要被"救济"的区域
# ============================================================

# ── Relief Vulnerability: 用 +10 策略作为基准 ──
best_strategy = [r for r in strategy_results if r["n_sites"] == 10]
if best_strategy:
    coverage_geom = best_strategy[0]["coverage_geom"]
    coverage_prep = shapely_prep(coverage_geom)

    grid["covered_by_10"] = grid.apply(
        lambda r: coverage_prep.intersects(Point(r["cx"], r["cy"])), axis=1
    )
    grid["relief_vulnerability"] = grid["gap_index"] * (~grid["covered_by_10"]).astype(float)

    uncovered_high = grid[(~grid["covered_by_10"]) & (grid["gap_index"] > grid["gap_index"].quantile(0.75))]
    print(f"=== Relief Vulnerability (+10 baseline) ===")
    print(f"Covered grids: {grid['covered_by_10'].sum():,} / {len(grid):,}")
    print(f"Uncovered + high gap: {len(uncovered_high):,} grids")
    print(f"Max vulnerability: {grid['relief_vulnerability'].max():.4f}")
else:
    grid["covered_by_10"] = False
    grid["relief_vulnerability"] = grid["gap_index"]
    print("No +10 strategy available")

# ══════════════════════════════════════════
#  保存所有输出
# ══════════════════════════════════════════

# 1) Gap Index 网格
grid.to_file(OUT / "sz_gap_grid.gpkg", driver="GPKG")
print(f"\nGap grid       -> data_out/sz_gap_grid.gpkg  ({len(grid):,} cells)")

# 2) 候选站点
site_pool.to_file(OUT / "sz_candidate_sites.gpkg", driver="GPKG")
print(f"Candidate sites -> data_out/sz_candidate_sites.gpkg  ({len(site_pool):,} sites)")

# 3) 策略覆盖圈
strategy_rows = []
for r in strategy_results:
    strategy_rows.append({
        "n_sites": r["n_sites"],
        "coverage_area_pct": round(r["coverage_area_pct"], 1),
        "demand_coverage_pct": round(r["demand_coverage_pct"], 1),
        "pop_coverage_pct": round(r["pop_coverage_pct"], 1),
        "geometry": r["coverage_geom"],
    })
strategy_gdf = gpd.GeoDataFrame(strategy_rows, crs=4326)
strategy_gdf.to_file(OUT / "sz_strategy_coverage.gpkg", driver="GPKG")
print(f"Strategy maps   -> data_out/sz_strategy_coverage.gpkg  ({len(strategy_gdf)} strategies)")

# 4) Top sites per strategy
for r in strategy_results:
    n = r["n_sites"]
    top_sites = site_pool[site_pool["site_id"].isin(r["site_ids"])]
    top_sites.to_file(OUT / f"sz_strategy_{n}_sites.gpkg", driver="GPKG")

print("\n=== Summary ===")
print(f"Gap index range: {grid['gap_index'].min():.4f} - {grid['gap_index'].max():.4f}")
print(f"Candidate sites: {len(site_pool)} (hub: {(site_pool['site_class']=='hub').sum()}, station: {(site_pool['site_class']=='station').sum()}, endpoint: {(site_pool['site_class']=='endpoint').sum()})")
print(f"\nStrategy comparison:")
for r in strategy_results:
    print(f"  +{r['n_sites']}: {r['demand_coverage_pct']:.1f}% demand covered, {r['pop_coverage_pct']:.1f}% population covered")